# Polynomial fit for the cross-section of electron-H atom elastic collisions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import k, e, pi, m_e, m_p, epsilon_0

In [2]:
# - Morgan database, www.lxcat.net, retrieved on May 3, 2025.
# SPECIES: e / H
# PROCESS: E + H -> E + H, Elastic
# PARAM.:  m/M = 0.000548193, complete set
# UPDATED: 2010-07-16 14:34:41

In [ ]:
# Log data
data = np.loadtxt( './H_cross_section.txt', skiprows=2)
E_eV = data[:,0]
sigma_m2 = data[:,1]

# Perform a polynomial fit of the log-transformed data
log_E_eV = np.log(E_eV[1:])
log_sigma_m2 = np.log(sigma_m2[1:])

# Fit a polynomial of degree 4
p = np.polyfit(log_E_eV, log_sigma_m2, 4)

def sigma_eh( E_eV ):
    """
    Calculate the cross-section for electron-H atom elastic collisions

    Parameters
    ----------
    E_eV : float
    """
    log_E_eV = np.log(E_eV)
    log_sigma_m2 = p[4] + p[3]*log_E_eV + p[2]*log_E_eV**2 + p[1]*log_E_eV**3 + p[0]*log_E_eV**4
    sigma_m2 = np.exp(log_sigma_m2)
    return sigma_m2

# Plot the cross-section
plt.loglog(data[:,0], data[:,1], '-o')
# Plot the polynomial fit
E_eV = np.logspace(np.log10(E_eV[1]), np.log10(E_eV[-1]), 100)
sigma_m2 = sigma_eh(E_eV)
plt.loglog(E_eV, sigma_m2, '-r')

plt.xlabel('Energy (eV)')
plt.ylabel('Cross-section (m^2)')
plt.grid()

In [ ]:
p

# Evaluate collision frequencies

In [5]:
def nu_eh( n_h, T_e ):
    """
    Calculate the collision frequency for electron-H atom elastic collisions
    using Eq. (A5) from https://arxiv.org/pdf/2305.16779

    Parameters
    ----------
    n_e : float
        Electron number density (m^-3)
    T_e : float
        Electron temperature (K)

    Returns
    -------
    nu_eH : float
        Collision frequency for electron-H atom elastic collisions (s^-1)
    """
    # Calculate mean energy
    E_bar = 4/np.pi * k * T_e

    # Calculate reduced mass
    mu = m_e * m_p / (m_e + m_p)

    # Find corresponding cross-section
    sigma = sigma_eh(E_bar/e) # Convert E_bar to eV

    # Calculate collision frequency
    nu_eh = n_h * 4/3 * np.sqrt(2*E_bar/mu) * sigma
    return nu_eh


In [6]:
def nu_ei( n_e, T_e, T_h ):
    """
    Calculate the collision frequency for electron-ion collisions
    using Eq. (A1) from https://arxiv.org/pdf/2305.16779

    Parameters
    ----------
    n_e : float
        Electron number density (m^-3)
    T_e : float
        Electron temperature (K)

    Returns
    -------
    nu_ei : float
        Collision frequency for electron-ion collisions (s^-1)
    """
    # Calculate the Coulomb logarithm
    Lambda_ei = np.log( 3/(2*np.sqrt(np.pi)) * np.sqrt(
        (4*pi*epsilon_0*k*T_e)**3 / (e**6 * n_e * (1+T_e/T_h))
    ))

    # Calculate the collision frequency
    nu_ei = 4/3 * np.sqrt(2*np.pi/m_e) * n_e * e**4 * Lambda_ei / (
        (4*pi*epsilon_0)**2 * (k*T_e)**(3/2)
    )
    return nu_ei


In [ ]:
T_e_eV = np.logspace(-1, 2, 1000)
T_e = T_e_eV * e/k
T_h = 300
n_h = 1e24 # m^-3
n_e = 1e23 # m^-3
nu_h = nu_eh(n_h, T_e)
nu_i = nu_ei(n_e, T_e, T_h)
plt.grid()

plt.loglog(T_e_eV, nu_h, label='electron-H')
plt.loglog(T_e_eV, nu_i, label='electron-ion')
plt.legend()
plt.xlabel('Electron temperature (K)')
plt.ylabel(r'Collision frequency (s$^{-1}$)')